In [ ]:
"""
=============================================================================
Machine Learning Analysis of CXCR1/CXCR2 Modulators
=============================================================================
"""

import os
import sys

# =====================================================================
# INSTALLATIONS & IMPORTS
# =====================================================================
try:
    import chembl_webresource_client
    import rdkit
    import imblearn
    import sklearn
    import cairosvg
except ImportError:
    os.system('pip install chembl_webresource_client rdkit imbalanced-learn scikit-learn cairosvg -q')
    import chembl_webresource_client
    import rdkit
    import imblearn
    import sklearn
    import cairosvg

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from IPython.display import display, Markdown

# Check versions
!python --version
print(f"CHEMBL version: {chembl_webresource_client.__version__}")
print(f"RDKit version: {rdkit.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Imbalanced-learn version: {imblearn.__version__}")
print(f"Pandas version: {pd.__version__}")

RDLogger.DisableLog('rdApp.*')

Python 3.12.13
CHEMBL version: 0.10.9
RDKit version: 2026.03.1
Scikit-learn version: 1.6.1
Imbalanced-learn version: 0.14.1
Pandas version: 2.2.2


In [ ]:
# CONFIGURATIONS
TARGETS = {'CXCR1': 'CHEMBL4029', 'CXCR2': 'CHEMBL2434'}
BINDING_KEYWORDS = 'binding|displacement|125I|radioligand|SPA|DELFIA|AF647'
FUNCTIONAL_KEYWORDS = 'beta-arrestin|recruitment|PathHunter|Tango|calcium|calicum|reporter gene|myeloperoxidase|FLIPR|CD11b|chemotaxis'
EXCLUDE_KEYWORDS = 'unknown origin'
CUSTOM_PALETTE = {'Binding Affinity': '#4C72B0', 'Functional Antagonism': '#C44E52'}

# =====================================================================
# 1. Programmatic retrieval of bioactivity data (IC50) from ChEMBL
# =====================================================================
print("--- 1. Fetching Data from ChEMBL ---")
activity = new_client.activity
all_data = []

for name, chembl_id in TARGETS.items():
    print(f"Fetching data for {name} ({chembl_id})...")
    res = activity.filter(target_chembl_id=chembl_id, standard_type="IC50")
    df = pd.DataFrame.from_dict(res)
    if not df.empty:
        df['Receptor'] = name
        print(f"Successfully downloaded {len(df)} IC50 records for {name}.")
        all_data.append(df)

master_df = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
master_df = master_df[['molecule_chembl_id', 'canonical_smiles', 'pchembl_value', 'assay_description', 'Receptor']].copy()
master_df.dropna(subset=['canonical_smiles', 'pchembl_value', 'assay_description'], inplace=True)
master_df['pchembl_value'] = pd.to_numeric(master_df['pchembl_value'], errors='coerce')
master_df.dropna(subset=['pchembl_value'], inplace=True)
print(f"Total usable records saved: {len(master_df)}")

# =====================================================================
# 2. Data curation and stratification into binding vs. functional
# =====================================================================
print("\n--- 2. Curation & Stratification ---")
def extract_modality(df, keyword_pattern):
    valid = df[~df['assay_description'].str.contains(EXCLUDE_KEYWORDS, case=False, na=False)]
    modality = valid[valid['assay_description'].str.contains(keyword_pattern, case=False, na=False)].copy()
    modality.sort_values('pchembl_value', ascending=False, inplace=True)
    modality.drop_duplicates('canonical_smiles', keep='first', inplace=True)
    return modality

cxcr1, cxcr2 = master_df[master_df['Receptor'] == 'CXCR1'], master_df[master_df['Receptor'] == 'CXCR2']

# =====================================================================
# 3. Calculation of receptor selectivity profiles (CXCR2 vs. CXCR1)
# =====================================================================
print("\n--- 3. Selectivity Calculation ---")
p_bind = pd.merge(extract_modality(cxcr1, BINDING_KEYWORDS), extract_modality(cxcr2, BINDING_KEYWORDS), on='canonical_smiles', suffixes=('_cxcr1', '_cxcr2'))
p_func = pd.merge(extract_modality(cxcr1, FUNCTIONAL_KEYWORDS), extract_modality(cxcr2, FUNCTIONAL_KEYWORDS), on='canonical_smiles', suffixes=('_cxcr1', '_cxcr2'))

for d in [p_bind, p_func]:
    d['Selectivity_Score'] = d['pchembl_value_cxcr2'] - d['pchembl_value_cxcr1']

print(f"Final Paired Binding Compounds: {len(p_bind)}")
print(f"Final Paired Functional Compounds: {len(p_func)}")

# =====================================================================
# 4. Bemis-Murcko scaffold extraction for chemotype analysis
# =====================================================================
print("\n--- 4. Scaffold Extraction ---")
def get_bm(s):
    try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(s)))
    except: return None

for df_temp in [p_bind, p_func]:
    df_temp['BM_Scaffold_SMILES'] = df_temp['canonical_smiles'].apply(get_bm)

# =====================================================================
# 5. Physicochemical feature extraction and Random Forest classification
# =====================================================================
print("\n--- 5. ML & Profiling ---")
def get_feats(s):
    m = Chem.MolFromSmiles(s)
    return pd.Series({
        'MolWt': Descriptors.MolWt(m), 'LogP': Descriptors.MolLogP(m),
        'TPSA': Descriptors.TPSA(m), 'NumHDonors': Descriptors.NumHDonors(m),
        'HBA': Descriptors.NumHAcceptors(m), 'Fsp3': Descriptors.FractionCSP3(m)
    })

p_bind['Label'], p_func['Label'] = 0, 1
combined = pd.concat([p_bind, p_func], ignore_index=True)
combined['Modality'] = combined['Label'].map({0: 'Binding Affinity', 1: 'Functional Antagonism'})

print("Calculating properties for Combined Dataset...")
feats = combined['canonical_smiles'].apply(get_feats)

X, y = feats.fillna(0), combined['Label']
model_pipe = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'))
])

cv_scores = cross_val_score(model_pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='roc_auc')
print(f"CV ROC-AUC: {cv_scores.mean():.3f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model_pipe.fit(X_train, y_train)
print(classification_report(y_test, model_pipe.predict(X_test)))

y_prob = model_pipe.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, y_prob)
print(f"Test Set ROC-AUC: {test_roc_auc:.3f}")


--- 1. Fetching Data from ChEMBL ---
Fetching data for CXCR1 (CHEMBL4029)...
Successfully downloaded 1284 IC50 records for CXCR1.
Fetching data for CXCR2 (CHEMBL2434)...
Successfully downloaded 2318 IC50 records for CXCR2.
Total usable records saved: 1419

--- 2. Curation & Stratification ---

--- 3. Selectivity Calculation ---
Final Paired Binding Compounds: 57
Final Paired Functional Compounds: 72

--- 4. Scaffold Extraction ---

--- 5. ML & Profiling ---
Calculating properties for Combined Dataset...
CV ROC-AUC: 0.976
              precision    recall  f1-score   support

           0       0.91      0.91      0.91        11
           1       0.93      0.93      0.93        15

    accuracy                           0.92        26
   macro avg       0.92      0.92      0.92        26
weighted avg       0.92      0.92      0.92        26

Test Set ROC-AUC: 0.982
